In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import mlflow
import sys
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports done!")

✅ Imports done!


In [2]:
sys.path.append('../utils')
from snowflake_connector import read_table

# Load from Snowflake
customer_features = read_table('GOLD_CUSTOMER_FEATURES')
catalog_df = read_table('SILVER_PRODUCT_CATALOG', schema='GOLD_SILVER')

print(f"\n✅ Loaded from Snowflake!")
print(f"Customer features: {customer_features.shape}")
print(f"Catalog: {catalog_df.shape}")
print(customer_features.columns.tolist())

✅ Loaded GOLD_CUSTOMER_FEATURES: 10,000 rows
✅ Loaded SILVER_PRODUCT_CATALOG: 8 rows

✅ Loaded from Snowflake!
Customer features: (10000, 30)
Catalog: (8, 7)
['CUSTOMER_ID', 'AGE', 'INCOME', 'CREDIT_SCORE', 'TENURE_MONTHS', 'SEGMENT', 'EMPLOYMENT_TYPE', 'MARITAL_STATUS', 'TOTAL_TRANSACTIONS', 'TOTAL_SPEND', 'AVG_TRANSACTION', 'MAX_TRANSACTION', 'TRAVEL_PCT', 'DINING_PCT', 'GROCERIES_PCT', 'RENT_PCT', 'SHOPPING_PCT', 'ENTERTAINMENT_PCT', 'HEALTHCARE_PCT', 'UTILITIES_PCT', 'NUM_PRODUCTS', 'HAS_CHECKING', 'HAS_SAVINGS', 'HAS_TRAVEL_CARD', 'HAS_CASHBACK_CARD', 'HAS_PERSONAL_LOAN', 'HAS_HOME_LOAN', 'HAS_INVESTMENT', 'HAS_FIXED_DEPOSIT', 'UPDATED_AT']


In [3]:
# Spending categories matching our Gold table columns
categories = ['TRAVEL_PCT', 'DINING_PCT', 'GROCERIES_PCT', 'RENT_PCT', 
              'SHOPPING_PCT', 'ENTERTAINMENT_PCT', 'HEALTHCARE_PCT', 'UTILITIES_PCT']

# ideal spending profile per product
product_profiles = {
    'checking_account':     [0.1, 0.1, 0.2, 0.3, 0.1, 0.1, 0.05, 0.05],
    'savings_account':      [0.1, 0.1, 0.2, 0.2, 0.1, 0.1, 0.1,  0.1],
    'travel_credit_card':   [0.4, 0.2, 0.1, 0.1, 0.1, 0.05,0.02, 0.03],
    'cashback_credit_card': [0.1, 0.2, 0.3, 0.1, 0.2, 0.05,0.02, 0.03],
    'personal_loan':        [0.1, 0.1, 0.2, 0.3, 0.1, 0.05,0.05, 0.1],
    'home_loan':            [0.05,0.1, 0.2, 0.4, 0.1, 0.05,0.05, 0.05],
    'investment_fund':      [0.2, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1,  0.1],
    'fixed_deposit':        [0.1, 0.1, 0.2, 0.1, 0.1, 0.1, 0.1,  0.2],
}

print("✅ Product profiles defined!")
for name in product_profiles:
    print(f"  {name}")

✅ Product profiles defined!
  checking_account
  savings_account
  travel_credit_card
  cashback_credit_card
  personal_loan
  home_loan
  investment_fund
  fixed_deposit


In [4]:
# Build product vector matrix
product_names = list(product_profiles.keys())
product_matrix = np.array(list(product_profiles.values()))

# Build customer spending vector matrix from Snowflake Gold
customer_matrix = customer_features[categories].values

# Calculate cosine similarity
similarity_matrix = cosine_similarity(customer_matrix, product_matrix)

# Convert to DataFrame
similarity_df = pd.DataFrame(
    similarity_matrix,
    columns=product_names,
    index=customer_features['CUSTOMER_ID']
)

print("✅ Cosine similarity calculated!")
print(f"Shape: {similarity_df.shape}")
print("\nTop products for C_00001:")
print(similarity_df.loc['C_00001'].sort_values(ascending=False))

✅ Cosine similarity calculated!
Shape: (10000, 8)

Top products for C_00001:
travel_credit_card      0.900909
investment_fund         0.750008
checking_account        0.529093
personal_loan           0.526301
savings_account         0.509119
home_loan               0.430987
fixed_deposit           0.425446
cashback_credit_card    0.369988
Name: C_00001, dtype: float64


In [5]:
# Get products each customer already owns from Gold table
owned_products = {}
for _, row in customer_features.iterrows():
    owned = set()
    if row['HAS_CHECKING'] == 1:    owned.add('checking_account')
    if row['HAS_SAVINGS'] == 1:     owned.add('savings_account')
    if row['HAS_TRAVEL_CARD'] == 1: owned.add('travel_credit_card')
    if row['HAS_CASHBACK_CARD'] == 1: owned.add('cashback_credit_card')
    if row['HAS_PERSONAL_LOAN'] == 1: owned.add('personal_loan')
    if row['HAS_HOME_LOAN'] == 1:   owned.add('home_loan')
    if row['HAS_INVESTMENT'] == 1:  owned.add('investment_fund')
    if row['HAS_FIXED_DEPOSIT'] == 1: owned.add('fixed_deposit')
    owned_products[row['CUSTOMER_ID']] = owned

# Generate top 3 recommendations per customer
cb_results = []
for customer_id in customer_features['CUSTOMER_ID']:
    owned = owned_products.get(customer_id, set())
    scores = similarity_df.loc[customer_id]
    unowned_scores = scores[~scores.index.isin(owned)]
    top3 = unowned_scores.sort_values(ascending=False).head(3)
    for product_name, score in top3.items():
        cb_results.append({
            'customer_id': customer_id,
            'product_name': product_name,
            'cb_score': round(score, 4)
        })

cb_df = pd.DataFrame(cb_results)

print("✅ Content-based recommendations generated!")
print(f"Total: {len(cb_df):,}")
print("\nSample:")
print(cb_df.head(9))

✅ Content-based recommendations generated!
Total: 29,966

Sample:
  customer_id        product_name  cb_score
0     C_00001  travel_credit_card    0.9009
1     C_00001     investment_fund    0.7500
2     C_00001       personal_loan    0.5263
3     C_00002     investment_fund    0.7316
4     C_00002     savings_account    0.4316
5     C_00002       personal_loan    0.4139
6     C_00003  travel_credit_card    0.8725
7     C_00003     investment_fund    0.6752
8     C_00003     savings_account    0.3749


In [6]:
# Set MLflow tracking
mlflow.set_tracking_uri('file:///C:/Users/Foram/Projects/Fintech Recommender System/fintech-recommender/mlflow')
mlflow.set_experiment("fintech_recommender_cb")

with mlflow.start_run(run_name="ContentBased_v2_snowflake"):
    mlflow.log_param("similarity_metric", "cosine")
    mlflow.log_param("data_source", "Snowflake Gold")
    mlflow.log_param("features_used", "spending_percentages")
    mlflow.log_metric("total_recommendations", len(cb_df))
    mlflow.log_metric("unique_customers", cb_df['customer_id'].nunique())

# Save recommendations
cb_df.to_csv('../data/cb_recommendations.csv', index=False)

print("✅ Content-based notebook complete!")
print(f"Total: {len(cb_df):,}")
print(f"Unique customers: {cb_df['customer_id'].nunique():,}")

2026/03/22 10:53:43 INFO mlflow.tracking.fluent: Experiment with name 'fintech_recommender_cb' does not exist. Creating a new experiment.


✅ Content-based notebook complete!
Total: 29,966
Unique customers: 10,000
